# Phase 4.1 — Pretrained OSNet Re-ID Inference Verification

**Goal:** Verify that a pretrained OSNet model can load and produce a valid appearance embedding from **one real TRACE person crop**.

This notebook does NOT:
- Process all 353 crops
- Perform cross-camera matching
- Train or fine-tune any model

**Runtime requirement:** GPU — `Runtime → Change runtime type → GPU`

## Step 1 — Verify GPU Environment

In [ ]:
import torch

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")
    device = torch.device('cuda')
else:
    print("WARNING: CUDA not available — running on CPU.")
    print("For Phase 4.1, GPU is strongly recommended.")
    print("Go to Runtime → Change runtime type → GPU and re-run.")
    device = torch.device('cpu')

print(f"\nUsing device    : {device}")
print("=" * 50)

## Step 2 — Install Dependencies

Install `torchreid` from source (the pip package can lag behind; installing from GitHub ensures latest pretrained weight support).

> **After this cell completes:** go to `Runtime → Restart session`, then re-run all cells from Step 1.

In [ ]:
# Install torchreid from the official KaiyangZhou repository
!pip install -q git+https://github.com/KaiyangZhou/deep-person-reid.git

# Also ensure Pillow and numpy are present (usually pre-installed in Colab)
!pip install -q Pillow numpy

# -------------------------------------------------------
# IMPORTANT: After this cell finishes, go to:
#   Runtime → Restart session
# Then run ALL cells again from Step 1.
# Colab requires a kernel restart to pick up newly installed
# packages when installed from source via pip.
# -------------------------------------------------------
print("Installation complete. Please restart the runtime now (Runtime → Restart session).")
print("After restart, run all cells from Step 1.")

## Step 3 — Import Libraries

In [ ]:
import os
import json
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
import torchvision.transforms as T

import torchreid

print("All imports successful.")
# torchreid installed from GitHub source may not expose __version__
torchreid_version = getattr(torchreid, '__version__', 'installed from source (no __version__)')
print(f"torchreid version : {torchreid_version}")

## Step 4 — Mount Google Drive and Locate TRACE Dataset

The TRACE project must be uploaded to your Google Drive, or you can upload the crops folder directly.  
Update `TRACE_ROOT` below to point to your project root inside Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------------------------------
# UPDATE THIS PATH to where TRACE lives in your Drive
# Example: '/content/drive/MyDrive/Trace'
# -------------------------------------------------------
TRACE_ROOT = '/content/drive/MyDrive/Trace'  # <-- adjust if needed

CROPS_DIR      = os.path.join(TRACE_ROOT, 'dataset', 'crops_phase3_final')
METADATA_PATH  = os.path.join(TRACE_ROOT, 'dataset', 'crops_metadata_phase3_final.json')

# Verify paths exist
assert os.path.isdir(CROPS_DIR), (
    f"Crops directory not found: {CROPS_DIR}\n"
    "Please update TRACE_ROOT above."
)
assert os.path.isfile(METADATA_PATH), (
    f"Metadata file not found: {METADATA_PATH}\n"
    "Please update TRACE_ROOT above."
)

print(f"Crops directory  : {CROPS_DIR}")
print(f"Metadata file    : {METADATA_PATH}")

# Count available crops
crop_files = sorted([
    f for f in os.listdir(CROPS_DIR)
    if f.lower().endswith('.jpg')
])
print(f"Crops found      : {len(crop_files)}")

## Step 5 — Select One Real TRACE Crop

Discovered from the actual directory — no hardcoded filename.

In [ ]:
# Load metadata
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"Metadata entries : {len(metadata)}")

# Pick the first entry from metadata (real crop with known track/frame info)
sample_entry = metadata[0]

# Build absolute path from the relative crop_path stored in metadata
relative_crop_path = sample_entry['crop_path']   # e.g. 'dataset/crops_phase3_final/C01_track1_frame0000.jpg'
crop_filename = os.path.basename(relative_crop_path)
CROP_PATH = os.path.join(CROPS_DIR, crop_filename)

assert os.path.isfile(CROP_PATH), f"Crop file not found: {CROP_PATH}"

print("\n" + "=" * 50)
print("SELECTED CROP")
print("=" * 50)
print(f"Path            : {CROP_PATH}")
print(f"Camera          : {sample_entry['camera_id']}")
print(f"Track ID        : {sample_entry['track_id']}")
print(f"Frame           : {sample_entry['frame']}")
print(f"Timestamp       : {sample_entry['timestamp']}")
print(f"Det. confidence : {sample_entry['detection_confidence']:.4f}")
print("=" * 50)

# Display the crop
from IPython.display import display
img_display = Image.open(CROP_PATH)
print(f"Original image size (W x H): {img_display.size}")
display(img_display)

## Step 6 — Load Pretrained OSNet Model

Using `osnet_x1_0` with pretrained ImageNet/Market-1501 weights via `torchreid`.  
The embedding dimension is verified programmatically — not assumed.

In [ ]:
# Build the model using torchreid's model factory
# num_classes=1000 loads the full classifier head; we will extract features
# before the classifier so the output is the embedding vector.
model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=1000,   # required by the builder; ignored during feature extraction
    pretrained=True     # downloads pretrained weights automatically
)

model.eval()
model = model.to(device)

print("Model loaded successfully.")
print(f"Model type      : {type(model).__name__}")
print(f"Device          : {next(model.parameters()).device}")

## Step 7 — Preprocess the Crop

OSNet was trained on person Re-ID datasets with:
- Input resolution: **256 × 128** (H × W)  
- ImageNet normalization: mean `[0.485, 0.456, 0.406]`, std `[0.229, 0.224, 0.225]`
- RGB channel order (not BGR)

PIL loads images in RGB by default, so no channel-swap needed.

In [ ]:
# OSNet expected input resolution (H x W)
INPUT_H = 256
INPUT_W = 128

# Standard ImageNet normalization used during OSNet pretraining
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = T.Compose([
    T.Resize((INPUT_H, INPUT_W)),          # resize to 256x128
    T.ToTensor(),                           # HWC uint8 [0,255] → CHW float [0.0,1.0]
    T.Normalize(mean=IMAGENET_MEAN,
                std=IMAGENET_STD),          # per-channel normalization
])

# Load image with PIL (guarantees RGB, not BGR)
img_pil = Image.open(CROP_PATH).convert('RGB')

# Apply transform
img_tensor = transform(img_pil)       # shape: [3, 256, 128]
img_tensor = img_tensor.unsqueeze(0)  # add batch dim → [1, 3, 256, 128]
img_tensor = img_tensor.to(device)

print("=" * 50)
print("PREPROCESSING")
print("=" * 50)
print(f"Input image mode       : {img_pil.mode}  (RGB confirmed)")
print(f"Original size (W x H)  : {img_pil.size}")
print(f"Resized to (H x W)     : {INPUT_H} x {INPUT_W}")
print(f"Input tensor shape     : {list(img_tensor.shape)}")
print(f"Input tensor dtype     : {img_tensor.dtype}")
print(f"Input tensor device    : {img_tensor.device}")
print(f"Tensor min / max       : {img_tensor.min().item():.4f} / {img_tensor.max().item():.4f}")
print("=" * 50)

## Step 8 — Run Inference (Forward Pass)

OSNet's `forward()` returns `(global_feat, logits)` in training mode but only `global_feat` in eval mode.  
We use `model.eval()` + `torch.no_grad()` and extract the feature tensor directly.

In [ ]:
model.eval()

with torch.no_grad():
    output = model(img_tensor)

# In eval mode torchreid OSNet returns the feature tensor directly.
# Guard against tuple returns (some model configs return (feat, logits)).
if isinstance(output, (tuple, list)):
    embedding = output[0]   # global feature is always first
else:
    embedding = output

print("=" * 50)
print("INFERENCE RESULT")
print("=" * 50)
print(f"Input crop      : {CROP_PATH}")
print(f"Device          : {device}")
print(f"Input tensor shape  : {list(img_tensor.shape)}")
print(f"Output shape        : {list(embedding.shape)}")
print(f"Embedding dtype     : {embedding.dtype}")
print(f"Embedding dimension : {embedding.shape[1]}")
print("=" * 50)

## Step 9 — Verify Embedding Quality

Checks: correct type, batch size 1, no NaN, no Inf, non-zero, convertible to NumPy, L2 norm.

In [ ]:
# Move to CPU for numpy operations
emb_cpu = embedding.cpu()

# --- Check 1: is a tensor ---
assert isinstance(emb_cpu, torch.Tensor), "FAIL: output is not a torch.Tensor"
print("[PASS] Output is a torch.Tensor")

# --- Check 2: batch size is 1 ---
assert emb_cpu.shape[0] == 1, f"FAIL: expected batch size 1, got {emb_cpu.shape[0]}"
print(f"[PASS] Batch size is 1")

# --- Check 3: embedding dimension reported ---
embed_dim = emb_cpu.shape[1]
print(f"[PASS] Embedding dimension: {embed_dim}")

# --- Check 4: no NaN ---
nan_count = torch.isnan(emb_cpu).sum().item()
assert nan_count == 0, f"FAIL: embedding contains {nan_count} NaN values"
print(f"[PASS] NaN values: {nan_count}")

# --- Check 5: no Inf ---
inf_count = torch.isinf(emb_cpu).sum().item()
assert inf_count == 0, f"FAIL: embedding contains {inf_count} Inf values"
print(f"[PASS] Inf values: {inf_count}")

# --- Check 6: non-zero ---
max_abs = emb_cpu.abs().max().item()
assert max_abs > 0, "FAIL: embedding is all zeros"
print(f"[PASS] Embedding is non-zero (max abs value: {max_abs:.6f})")

# --- Check 7: convertible to NumPy ---
emb_numpy = emb_cpu.numpy()
assert isinstance(emb_numpy, np.ndarray), "FAIL: could not convert to NumPy array"
print(f"[PASS] Converted to NumPy array, shape: {emb_numpy.shape}")

# --- Check 8: L2 norm ---
l2_norm = np.linalg.norm(emb_numpy[0])
print(f"[PASS] L2 norm: {l2_norm:.6f}")

print()
print("=" * 50)
print("EMBEDDING SUMMARY")
print("=" * 50)
print(f"Embedding shape     : {emb_numpy.shape}")
print(f"Embedding dtype     : {emb_numpy.dtype}")
print(f"NaN values          : {nan_count}")
print(f"Inf values          : {inf_count}")
print(f"L2 norm             : {l2_norm:.6f}")
print("=" * 50)

print("\nFirst 10 embedding values:")
print(emb_numpy[0][:10])

## Step 10 — Save Test Artifact

Save the single embedding as a temporary `.npy` file in `/content/`.  
This is a Colab-local artifact and is NOT committed to the project repository.

In [ ]:
SAVE_PATH = '/content/osnet_test_embedding.npy'

np.save(SAVE_PATH, emb_numpy)

# Reload and verify round-trip integrity
emb_reloaded = np.load(SAVE_PATH)
assert np.allclose(emb_numpy, emb_reloaded), "FAIL: saved/reloaded embedding does not match"

print(f"Embedding saved to  : {SAVE_PATH}")
print(f"Saved shape         : {emb_reloaded.shape}")
print(f"Round-trip check    : PASS (saved == reloaded)")

## Step 11 — Optional: Determinism Check

Run the same image through OSNet a second time and verify the two embeddings are numerically identical.  
Expected cosine similarity: **≈ 1.0**  

> This checks *deterministic inference*, NOT Re-ID accuracy.

In [ ]:
model.eval()

with torch.no_grad():
    output2 = model(img_tensor)

if isinstance(output2, (tuple, list)):
    embedding2 = output2[0]
else:
    embedding2 = output2

emb2_cpu = embedding2.cpu()

# Cosine similarity between the two runs
cos_sim = F.cosine_similarity(emb_cpu, emb2_cpu, dim=1).item()

# Max absolute difference
max_diff = (emb_cpu - emb2_cpu).abs().max().item()

print("=" * 50)
print("DETERMINISM CHECK")
print("=" * 50)
print(f"Run 1 embedding shape  : {list(emb_cpu.shape)}")
print(f"Run 2 embedding shape  : {list(emb2_cpu.shape)}")
print(f"Cosine similarity      : {cos_sim:.8f}")
print(f"Max absolute diff      : {max_diff:.2e}")

if cos_sim > 0.9999:
    print("[PASS] Inference is deterministic (cosine similarity ≈ 1.0)")
else:
    print(f"[WARN] Cosine similarity is {cos_sim:.6f} — check for dropout/BatchNorm in train mode")
print("=" * 50)

## Step 12 — Final Report

In [ ]:
inference_pass = (
    nan_count == 0
    and inf_count == 0
    and max_abs > 0
    and isinstance(emb_numpy, np.ndarray)
)

print("=" * 60)
print("PHASE 4.1 — FINAL REPORT")
print("=" * 60)
print(f"Model               : OSNet x1_0")
print(f"Pretrained          : Yes")
print(f"Training performed  : No")
print(f"Device              : {device}")
print(f"GPU                 : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A (CPU)'}")
print(f"Input crop          : {CROP_PATH}")
print(f"Input shape         : {list(img_tensor.shape)}")
print(f"Output shape        : {list(embedding.shape)}")
print(f"Embedding dimension : {embed_dim}")
print(f"Embedding dtype     : {emb_numpy.dtype}")
print(f"NaN                 : {nan_count}")
print(f"Inf                 : {inf_count}")
print(f"L2 norm             : {l2_norm:.6f}")
print(f"Inference           : {'PASS' if inference_pass else 'FAIL'}")
print("=" * 60)
print()
if inference_pass:
    print("Phase 4.1 is COMPLETE — pretrained OSNet successfully generated")
    print("an embedding from one real TRACE person crop.")
else:
    print("Phase 4.1 FAILED — review the checks above.")